# Data Preparation: Bike Sharing

This notebook uses the insights derived from EDA to preprocess the bike sharing dataset, split the data, and upload it to HuggingFace.


In [ ]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), "../.."))

In [ ]:
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from src.preprocess import add_cyclic_hour, to_tensors
from src.data import save_to_hf


## Load Data

In [ ]:
bike_sharing = fetch_ucirepo(id=275)

X = bike_sharing.data.features.copy()
y = bike_sharing.data.targets.copy()

print(f'Raw dataset: X={X.shape}, y={y.shape}')


Raw dataset: X=(17379, 13), y=(17379, 1)


## Preprocessing

### Remove rare `weathersit == 4` records

In [ ]:
mask = X['weathersit'] != 4
X, y = X[mask].reset_index(drop=True), y[mask].reset_index(drop=True)
print(f'After filter: {X.shape}')


After filter: (17376, 13)


### Drop leakage, ID, and low-value columns

In [5]:
DROP_COLS = ['dteday', 'temp', 'holiday', 'weekday', 'mnth']

X.drop(columns=DROP_COLS, inplace=True)
print('Remaining features:', X.columns.tolist())


Remaining features: ['season', 'yr', 'hr', 'workingday', 'weathersit', 'atemp', 'hum', 'windspeed']


## Train / Test Split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')


Train: (13900, 8)  |  Test: (3476, 8)


## Transformation Pipeline

All transformations are **fit on the training set only**, then applied identically to the test set to prevent data leakage.


### 1 · Log-transform the target

In [7]:
y_train_log = np.log1p(y_train.values.flatten())
y_test_log  = np.log1p(y_test.values.flatten())


### 2 · Cyclical encoding for `hr`

`src.preprocess.add_cyclic_hour` takes the hour column and returns `(sin_hour, cos_hour)`.


In [ ]:
for df in [X_train, X_test]:
    df['hr_sin'], df['hr_cos'] = add_cyclic_hour(df['hr'])
    df.drop(columns=['hr'], inplace=True)


### 3 · One-hot encoding for `season` & `weathersit`

In [ ]:
OHE_COLS = ['season', 'weathersit']

X_train = pd.get_dummies(X_train, columns=OHE_COLS, drop_first=True)
X_test  = pd.get_dummies(X_test,  columns=OHE_COLS, drop_first=True)

# Align columns — handles any unseen category in test
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# Cast bool columns to int
for df in [X_train, X_test]:
    bool_cols = df.select_dtypes('bool').columns
    df[bool_cols] = df[bool_cols].astype(int)

print('Feature columns:', X_train.columns.tolist())
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')


Feature columns: ['yr', 'workingday', 'atemp', 'hum', 'windspeed', 'hr_sin', 'hr_cos', 'season_2', 'season_3', 'season_4', 'weathersit_2', 'weathersit_3']
Train: (13900, 12)  |  Test: (3476, 12)


### 4 · StandardScaler on continuous features

In [10]:
NUM_FEATURES = ['atemp', 'hum', 'windspeed']

scaler = StandardScaler()
X_train[NUM_FEATURES] = scaler.fit_transform(X_train[NUM_FEATURES])
X_test[NUM_FEATURES]  = scaler.transform(X_test[NUM_FEATURES])

print('Scaling complete.')
X_train.describe()


Scaling complete.


,yr,workingday,atemp,hum,windspeed,hr_sin,hr_cos,season_2,season_3,season_4,weathersit_2,weathersit_3
count,13900.000000,13900.000000,1.390000e+04,1.390000e+04,1.390000e+04,13900.000000,1.390000e+04,13900.000000,13900.000000,13900.000000,13900.000000,13900.000000
mean,0.501871,0.681295,7.412136e-17,-2.121405e-16,2.739935e-16,-0.004848,-3.754404e-03,0.254245,0.259137,0.244317,0.260288,0.083094
std,0.500014,0.465991,1.000036e+00,1.000036e+00,1.000036e+00,0.706580,7.076578e-01,0.435451,0.438177,0.429697,0.438807,0.276033
min,0.000000,0.000000,-2.771740e+00,-3.253639e+00,-1.542512e+00,-1.000000,-1.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,-8.312787e-01,-7.652674e-01,-6.921708e-01,-0.707107,-7.071068e-01,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,1.000000,5.074933e-02,1.234862e-02,3.611152e-02,0.000000,-1.836970e-16,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1.000000,1.000000,8.448657e-01,7.899646e-01,5.219043e-01,0.707107,7.071068e-01,1.000000,1.000000,0.000000,1.000000,0.000000
max,1.000000,1.000000,2.961733e+00,1.930468e+00,5.379832e+00,1.000000,1.000000e+00,1.000000,1.000000,1.000000,1.000000,1.000000


## Sanity-check Tensors

`src.preprocess.to_tensors` takes numpy arrays and returns `(X_tensor, y_tensor)`. The target is 1-D from `to_tensors`; we unsqueeze to `(N, 1)` to match the model's expected shape.


In [11]:
X_tr, y_tr = to_tensors(X_train.values, y_train_log)
X_te, y_te = to_tensors(X_test.values,  y_test_log)

y_tr = y_tr.unsqueeze(1)   # (N,) → (N, 1)
y_te = y_te.unsqueeze(1)

print(f'X_tr: {tuple(X_tr.shape)}  y_tr: {tuple(y_tr.shape)}')
print(f'X_te: {tuple(X_te.shape)}  y_te: {tuple(y_te.shape)}')


X_tr: (13900, 12)  y_tr: (13900, 1)
X_te: (3476, 12)  y_te: (3476, 1)


## Upload to Hugging Face

`src.data.save_to_hf` accepts a single DataFrame and an optional `split` argument (default `'train'`). We call it twice — once per split.


In [12]:
from huggingface_hub import login
login()


In [13]:
REPO = 'b-fatma/bike-sharing-federated'

save_to_hf(
    X_train=X_train,
    X_test=X_test,
    y_train_log=y_train_log,
    y_test_log=y_test_log,
    repo_name=REPO
)

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploaded dataset to Hugging Face: b-fatma/bike-sharing-federated
Train shape: (13900, 13)
Test shape: (3476, 13)
